# Señal y ruido

Por qué predecir no es ajustar, y qué hace falta suponer para poder hacerlo

## Por qué estamos aquí

Este curso trata de un problema muy concreto: dado un conjunto de observaciones pasadas, predecir el valor de una magnitud numérica en observaciones que **todavía no hemos visto**. El precio de un piso que aún no se ha alquilado, la demanda de la semana que viene, el coste de un siniestro que aún no se ha declarado.

Conviene decir desde el primer día en qué se diferencia esto de lo que han visto en econometría. Allí la pregunta es *¿cuánto vale $w_j$ y es significativo?*. Aquí la pregunta es *¿qué $\boldsymbol{w}$ predice mejor fuera de la muestra?*. Son preguntas distintas, con respuestas distintas, y a veces el mejor modelo para una es un mal modelo para la otra. Escríbanlo en el margen:

> Un modelo que predice bien no explica nada por el hecho de predecir bien.

En este capítulo demostraremos dos cosas. La primera, que **ajustar perfectamente los datos de entrenamiento siempre es posible y por tanto no significa nada** ([Teorema 1](#thm-interpolacion)). La segunda, que en cuanto uno se atreve a suponer algo sobre el ruido, el problema de modelizar se convierte en un problema de **verosimilitud**, que es un problema que sabemos resolver.

## Datos, señal y ruido

<span class="theorem-title">**Definición 1 (Datos y problema de predicción) **</span>Un conjunto de datos es $\mathcal{D}= \{(\mathbf{x}_i, y_i)\}_{i=1}^{n}$ con $\mathbf{x}_i \in \mathbb{R}^{p}$ y $y_i\in \mathbb{R}$. El problema de predicción consiste en elegir una función $f: \mathbb{R}^{p} \to \mathbb{R}$ que prediga bien $y$ en observaciones **no vistas**.

La palabra que hace todo el trabajo en [Definición 1](#def-datos) es *no vistas*. Sin ella el problema es trivial, como veremos enseguida.

In [ ]:
import torch
from matplotlib import pyplot as plt

kw_puntos = dict(color="black", facecolors="none", s=40, alpha=0.6, label="datos")

In [ ]:
# TODO: completar en clase

Figura 1: Descomposición de un conjunto de datos (puntos) en una señal subyacente (línea discontinua) y ruido (segmentos naranjas).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x.numpy(), y.numpy(), **kw_puntos)
ax.plot(x.numpy(), senal.numpy(), color="black", linestyle="--",
        label=r"Señal: $f(x_i) = 2x_i + 1$")
for i in range(n):
    etiqueta = r"Ruido: $\epsilon_i$" if i == 0 else None
    ax.plot([x[i].item()] * 2, [senal[i].item(), y[i].item()],
            color="#ff5700", alpha=0.5, linewidth=0.9, label=etiqueta)
ax.set(xlabel="$x$", ylabel="$y$")
ax.legend()
plt.show()

<span class="theorem-title">**Definición 2 (Modelo señal + ruido) **</span>$$
y_i= f(\mathbf{x}_i) + \varepsilon_i, \qquad \varepsilon_i \text{ iid}, \quad \mathbb{E}\!\left[ \varepsilon_i \right] = 0.
 \qquad(1)$$

A $f$ la llamamos **señal** y a $\varepsilon_i$ **ruido**.

Lo que queremos aprender es $f$, **no** los valores $y_i$. La distinción parece pedante y es el capítulo entero.

<span class="theorem-title">**Definición 3 (Error de entrenamiento y error de test) **</span>Dado un conjunto $\mathcal{S}$ y un modelo ajustado $\hat{f}$, $$
\hat{R}_{\mathcal{S}}(\hat{f}) = \frac{1}{\left\lvert \mathcal{S} \right\rvert} \sum_{i \in \mathcal{S}} \left( y_i- \hat{f}(\mathbf{x}_i) \right)^2 .
$$

> **Note**
>
> De momento el cuadrado es una elección arbitraria: lo estamos **midiendo**, no justificando. En [Sección 6](#sec-mle-teorema) demostraremos que se deduce de suponer ruido gaussiano.

## Interpolar no es aprender

<span class="theorem-title">**Definición 4 (Sobreajuste) **</span>Se produce **sobreajuste** cuando aumentar la complejidad del modelo mejora su error de entrenamiento y **empeora** su error de test.

El siguiente resultado es la razón de ser de la asignatura. No aparece en la mayoría de los cursos introductorios, y es de una línea.

<span class="theorem-title">**Teorema 1 (Interpolación exacta) **</span>Sean $x_1, \dots, x_{n} \in \mathbb{R}$ **distintos dos a dos** y sean $y_1, \dots, y_{n} \in \mathbb{R}$ cualesquiera. Entonces existe un polinomio $P$ de grado $\leq n- 1$ tal que $P(x_i) = y_i$ para todo $i$; es decir, con error de entrenamiento **exactamente cero**.

<span class="proof-title">*Prueba*. </span>Construimos la solución. Para cada $j$ defínase la *base de Lagrange* $$
\ell_j(x) = \prod_{k \neq j} \frac{x - x_k}{x_j - x_k},
$$ que está bien definida porque los $x_k$ son distintos. Evaluando, si $i = j$ todos los factores valen $1$, y si $i \neq j$ el factor $k = i$ se anula; por tanto $\ell_j(x_i) = \delta_{ij}$. Tomando $$
P(x) = \sum_{j=1}^{n} y_j\, \ell_j(x)
$$ se obtiene $P(x_i) = \sum_j y_j \delta_{ij} = y_i$. Cada $\ell_j$ tiene grado $n- 1$, luego $P$ también. $\square$

> **La consecuencia, dicha en voz alta**
>
> Un error de entrenamiento de cero **no es evidencia de nada**. Siempre se puede conseguir. Cualquier titular de la forma *«nuestro modelo acierta el 100 %»* que no diga sobre qué datos se ha medido es, en el mejor de los casos, ruido.

Veámoslo. Interpolamos los 20 puntos y después generamos datos nuevos con la **misma señal y ruido distinto**.

Figura 2: El mismo interpolador: perfecto a la izquierda, inservible a la derecha.

In [ ]:
from scipy import interpolate

f_int = interpolate.interp1d(x.numpy(), y.numpy(), kind="cubic")
x_denso = torch.linspace(0, 10, 200)
y_int = f_int(x_denso.numpy())

fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
ax[0].scatter(x.numpy(), y.numpy(), **kw_puntos)
ax[0].plot(x_denso.numpy(), y_int, color="#ff5700", linestyle="--", label="interpolador")
ax[0].set(title="Datos de entrenamiento", xlabel="$x$", ylabel="$y$")
ax[0].legend()

y_nuevo = 2.0 * x_denso + 1.0 + torch.randn(200) * 3.0
ax[1].scatter(x_denso.numpy(), y_nuevo.numpy(), **kw_puntos)
ax[1].plot(x_denso.numpy(), y_int, color="#ff5700", linestyle="--", label="interpolador")
ax[1].set(title="Datos nuevos, misma señal", xlabel="$x$")
ax[1].legend()
plt.show()

## Modelizar el ruido

Si queremos aprender $f$ en [Ecuación 1](#eq-senal-ruido), necesitamos poder hablar matemáticamente de $\varepsilon_i$. Supondremos que el ruido es gaussiano.

<span class="theorem-title">**Definición 5 (Densidad normal) **</span>$$
p(x; \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left( -\frac{(x-\mu)^2}{2\sigma^2} \right).
 \qquad(2)$$

<span class="theorem-title">**Lema 1 (Constante de normalización) **</span>$\displaystyle \int_{-\infty}^{\infty} e^{-z^2/2}\, \mathrm{d}z = \sqrt{2\pi}$.

## No lo demostramos

La demostración usa coordenadas polares sobre el cuadrado de la integral. Es bonita y no enseña nada de aprendizaje automático. Se acepta y se sigue.

<span class="theorem-title">**Teorema 2 (Traslación) **</span>Si $\varepsilon \sim \mathcal{N}(\mu, \sigma^2)$ y $c \in \mathbb{R}$, entonces $\varepsilon + c \sim \mathcal{N}(\mu + c, \sigma^2)$.

<span class="proof-title">*Prueba*. </span>Sea $W = \varepsilon + c$. Entonces $\mathbb{P}\!\left( W \leq t \right) = \mathbb{P}\!\left( \varepsilon \leq t - c \right) = \int_{-\infty}^{t-c} p(x;\mu,\sigma)\,\mathrm{d}x$. El cambio de variable $u = x + c$ (con $\mathrm{d}u = \mathrm{d}x$) transforma la integral en $\int_{-\infty}^{t} p(u - c;\mu,\sigma)\,\mathrm{d}u$, y por [Ecuación 2](#eq-densidad-normal) $p(u-c;\mu,\sigma) = p(u; \mu+c, \sigma)$. $\square$

<span class="theorem-title">**Teorema 3 (Escala) **</span>Si $\varepsilon \sim \mathcal{N}(\mu, \sigma^2)$ y $a \neq 0$, entonces $a\varepsilon \sim \mathcal{N}(a\mu, a^2\sigma^2)$.

<span class="proof-title">*Prueba*. </span>

$\square$

<span class="theorem-title">**Corolario 1 (Estandarización) **</span>Si $\varepsilon \sim \mathcal{N}(\mu, \sigma^2)$ entonces $Z = (\varepsilon - \mu)/\sigma\sim \mathcal{N}(0,1)$.

<span class="proof-title">*Prueba*. </span>Aplicar [Teorema 3](#thm-escala) con $a = 1/\sigma$ y después [Teorema 2](#thm-traslacion) con $c = -\mu/\sigma$. $\square$

<span class="theorem-title">**Teorema 4 (Media y varianza) **</span>Si $\varepsilon \sim \mathcal{N}(\mu, \sigma^2)$ entonces $\mathbb{E}\!\left[ \varepsilon \right] = \mu$ y $\mathrm{Var}\!\left( \varepsilon \right) = \sigma^2$.

<span class="proof-title">*Prueba*. </span>

$\square$

<span class="theorem-title">**Corolario 2 (Del modelo de ruido a la distribución del objetivo) **</span>$\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$ si y solo si $y_i\,\vert\,\mathbf{x}_i \sim \mathcal{N}(f(\mathbf{x}_i), \sigma^2)$.

<span class="proof-title">*Prueba*. </span>[Teorema 2](#thm-traslacion) con $c = f(\mathbf{x}_i)$. $\square$

Este corolario de una línea es la bisagra del curso: convierte una **hipótesis sobre los errores** en una **distribución para los datos**, y una distribución para los datos es algo a lo que se le puede calcular la verosimilitud.

## Verosimilitud

<span class="theorem-title">**Definición 6 (Modelo lineal-gaussiano) **</span>Con $p= 1$: $y_i\sim \mathcal{N}(w_0+ w_1 x_i,\ \sigma^2)$, de forma independiente.

> **La hipótesis que casi nadie enuncia**
>
> *Independientes* aquí significa condicionalmente a las $\mathbf{x}_i$. Es una hipótesis, no una verdad. En la **?@sec-grupos** del capítulo 6 veremos qué pasa cuando es falsa: un mismo anfitrión con doce anuncios en Madrid la rompe.

<span class="theorem-title">**Definición 7 (Verosimilitud y log-verosimilitud) **</span>$$
L(\boldsymbol{w}) = \prod_{i=1}^{n} p\left(y_i;\ f(\mathbf{x}_i),\ \sigma\right),
\qquad
\ell(\boldsymbol{w}) = \sum_{i=1}^{n} \log p\left(y_i;\ f(\mathbf{x}_i),\ \sigma\right).
$$

<span class="theorem-title">**Lema 2 (El logaritmo preserva el argmax) **</span>Si $g$ es estrictamente creciente, $\mathop{\mathrm{arg\,max}}_{\boldsymbol{w}} g(L(\boldsymbol{w})) = \mathop{\mathrm{arg\,max}}_{\boldsymbol{w}} L(\boldsymbol{w})$. En particular $\mathop{\mathrm{arg\,max}}L= \mathop{\mathrm{arg\,max}}\ell= \mathop{\mathrm{arg\,min}}(-\ell)$.

<span class="proof-title">*Prueba*. </span>Si $\boldsymbol{w}^{\star}$ maximiza $L$ entonces $L(\boldsymbol{w}) \leq L(\boldsymbol{w}^\star)$ para todo $\boldsymbol{w}$, y aplicando $g$ creciente, $g(L(\boldsymbol{w})) \leq g(L(\boldsymbol{w}^\star))$. El recíproco es idéntico usando que $g$ es estrictamente creciente. $\square$

<span class="theorem-title">**Definición 8 (Estimación máximo-verosímil) **</span>$\hat{\boldsymbol{w}}= \mathop{\mathrm{arg\,max}}_{\boldsymbol{w}} L(\boldsymbol{w}) = \mathop{\mathrm{arg\,min}}_{\boldsymbol{w}} \left(-\ell(\boldsymbol{w})\right)$.

> **Convenio: siempre minimizamos**
>
> Aunque el problema nazca como una maximización, en este curso **siempre** lo escribiremos como minimización. Es el convenio de toda la literatura de optimización y evita errores de signo en el capítulo 3.

### ¿Y ahora cómo lo resolvemos?

Por fuerza bruta, para ver que no escala.

In [ ]:
# TODO: completar en clase

In [ ]:
rejilla_w0 = torch.linspace(-5, 5, 60)
rejilla_w1 = torch.linspace(-1, 3, 60)
L = torch.tensor([[log_verosimilitud(a, b, x, y) for b in rejilla_w1] for a in rejilla_w0])

fig, ax = plt.subplots(figsize=(5.5, 4))
cs = ax.contourf(rejilla_w1.numpy(), rejilla_w0.numpy(), L.numpy(), levels=30)
i, j = divmod(int(L.argmax()), L.shape[1])
ax.scatter([rejilla_w1[j]], [rejilla_w0[i]], color="#ff5700", s=60, label="máximo en la rejilla")
ax.scatter([2.0], [1.0], color="white", marker="x", s=60, label="verdad")
ax.set(xlabel="$w_1$", ylabel="$w_0$")
ax.legend()
plt.colorbar(cs, ax=ax, label=r"$\ell(w)$")
plt.show()

Con dos parámetros y 60 valores cada uno son 3.600 evaluaciones. Con $p= 20$ parámetros serían $60^{20}$, más que átomos hay en la Vía Láctea. **Hace falta cálculo**, y eso es el capítulo 3.

## La trampa de esta semana

## «Mi $R^2$ de entrenamiento es 0,999»

Es exactamente [Teorema 1](#thm-interpolacion) en acción. Añadan grados de libertad suficientes y cualquiera consigue ese número. La pregunta correcta no es *cuánto ajusta* sino *sobre qué datos se ha medido*. Volveremos sobre esto con el aparato completo en el capítulo 6.

## Notación ↔ código

| Matemáticas       | Código            | Nota                    |
|-------------------|-------------------|-------------------------|
| $n$               | `n`, `X.shape[0]` | número de observaciones |
| $p$               | `p`, `X.shape[1]` | número de variables     |
| $\boldsymbol{w}$  | `w`               | vector de coeficientes  |
| $\sigma$          | `sigma`           | escala del ruido        |
| $f(\mathbf{x}_i)$ | `f(x)`            | la señal                |
| $\varepsilon_i$   | `ruido`           | lo que no explicamos    |

## Ejercicios

<span class="theorem-title">**Ejercicio 1 **</span>Sean los puntos $(1, 3)$, $(2, 5)$, $(4, 2)$. Construya explícitamente el polinomio interpolador de Lagrange y compruebe que pasa por los tres puntos.

## Solución del [Ejercicio 1](#exr-lagrange)

<span class="theorem-title">**Ejercicio 2 **</span>Tres titulares de prensa afirman: (a) «el modelo predice el impago con un 99 % de acierto»; (b) «hemos reducido el error a cero»; (c) «validado sobre 10 millones de registros». Para cada uno, diga qué información falta para poder juzgarlo.

<span class="theorem-title">**Ejercicio 3 **</span>Suponga $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$. Escriba la log-verosimilitud de una única observación y dibuje, sin calcular nada, cómo cambia al alejarse $y_i$ de $f(\mathbf{x}_i)$.

## Qué llevarte

-   Error de entrenamiento cero se consigue **siempre** ([Teorema 1](#thm-interpolacion)) y por tanto no es evidencia de nada.
-   Suponer una distribución para el ruido convierte el problema en uno de **verosimilitud** ([Corolario 2](#cor-target)).
-   Maximizar la verosimilitud por fuerza bruta no escala. Necesitamos gradientes.